# FrugalGPT — 5-Service LLM Cascade on HEADLINES

Cascade over the **5 services with full cached coverage** (~10,001 entries each) in
`db/HEADLINES.sqlite`:

| Service | Provider |
|---|---|
| `textsynth/gptj_6B` | Textsynth |
| `textsynth/fairseq_gpt_13B` | Textsynth |
| `ai21/j1-large` | AI21 |
| `openaichat/gpt-3.5-turbo` | OpenAI |
| `openaichat/gpt-4` | OpenAI |

Everything else in the original 12-API set (Cohere, the other AI21 tiers, GPT-Neox, etc.) is only
spot-cached on this dataset, so this notebook sticks to just these 5 — all cache hits, **no real
API keys** needed.

Trained on **2,000** rows, evaluated on **2,000** rows (kept smaller than the full run since the
point here is the *shape* of the 5-service tradeoff curve, not a Table-3-comparable number).

## 1. Clone and install

In [1]:
!git clone -q https://github.com/stanford-futuredata/FrugalGPT
%cd FrugalGPT
!pip install -q .
# setup.py doesn't list google-generativeai, but modelservice.py imports+configures it
# at MODULE LEVEL, so importing FrugalGPT breaks without this installed.
# Patch scoring.py for transformers 5.x compatibility (repo targets an older transformers API)
path = "src/FrugalGPT/scoring.py"
with open(path) as f:
    lines = f.readlines()

lines = [l for l in lines if "logging_dir=" not in l]       # arg removed in transformers 5.x
content = "".join(lines).replace("evaluation_strategy=", "eval_strategy=")  # arg renamed

with open(path, "w") as f:
    f.write(content)

!pip install -q .   # reinstall with the patched source
print("patched and reinstalled")
!pip install -q google-generativeai


/content/FrugalGPT
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.5/370.5 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 111.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires tenacity<10,>=9, but you have tenacity 8.5.0 which is incompatible.
  Preparing metadata (setup.py) ... done
patched and reinstalled


## 2. Download the response cache

In [2]:
import os
os.makedirs("db", exist_ok=True)
if not os.path.exists("db/HEADLINES.sqlite"):
    !wget -q -P db/ https://github.com/lchen001/DataHolder/releases/download/v0.0.1/HEADLINES.sqlite
if not os.path.exists("db/qa_cache.sqlite"):
    !wget -q -P db/ https://github.com/lchen001/DataHolder/releases/download/v0.0.1/qa_cache.sqlite
print("HEADLINES.sqlite ready:", os.path.exists("db/HEADLINES.sqlite"))


HEADLINES.sqlite ready: True


## 3. Placeholder keys (dummy strings — required for import, not for cached calls)

In [3]:
os.environ['OPENAI_API_KEY'] = 'OPENAI_API_KEY'
os.environ['AI21_STUDIO_API_KEY'] = 'AI21_STUDIO_API_KEY'
os.environ['COHERE_STUDIO_API_KEY'] = 'COHERE_STUDIO_API_KEY'
os.environ['TEXTSYNTH_API_SECRET_KEY'] = 'TEXTSYNTH_API_SECRET_KEY'
os.environ['ANTHROPIC_API_KEY'] = 'ANTHROPIC_API_KEY'
os.environ['GEMINI_API_KEY'] = 'GEMINI_API_KEY'
os.environ['TOGETHER_API_KEY'] = 'TOGETHER_API_KEY'


## 4. Load training/test data

In [4]:
import sys, copy
sys.path.append("src/")
import FrugalGPT

N_TRAIN = 2000
N_EVAL = 2000

train_raw = FrugalGPT.loadcsvdata("data/HEADLINES/train.csv")[:N_TRAIN]
test_raw = FrugalGPT.loadcsvdata("data/HEADLINES/test.csv")[:N_EVAL]
prefix = open('config/prompt/HEADLINES/prefix_e8.txt').read()
train_data = FrugalGPT.formatdata(train_raw, prefix)
test_data = FrugalGPT.formatdata(test_raw, prefix)
print("training on", len(train_data), "queries, evaluating on", len(test_data))


/content/FrugalGPT/src/service/utils.py:138: SyntaxWarning: invalid escape sequence '\('
  a1 = re.findall('\([a-zA-Z]\)', text)
/content/FrugalGPT/src/service/utils.py:143: SyntaxWarning: invalid escape sequence '\('
  return re.findall('\([a-zA-Z]\)', text)[-1]
/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/content/FrugalGPT/src/FrugalGPT/scoring.py:79: SyntaxWarning: invalid escape sequence '\('
  a1 = re.findall('\([a-zA-Z]\)', text)
/content/FrugalGPT/src/FrugalGPT/scoring.py:84: SyntaxWarning: invalid escape sequence '\('
  return re.findall('\([a-zA-Z]\)', text)[-1]


training on 2000 queries, evaluating on 2000


## 5. Score each of the 5 services alone

Gives us the individual LLM accuracy/cost points to overlay on the cascade curve later
(like the scatter points behind the red FrugalGPT line in the paper's Figure 5).

In [5]:
service_names = [
    'textsynth/gptj_6B',
    'textsynth/fairseq_gpt_13B',
    'ai21/j1-large',
    'openaichat/gpt-3.5-turbo',
    'openaichat/gpt-4',
]

genparams = FrugalGPT.GenerationParameter(max_tokens=50, temperature=0.1, stop=['\n'])
MyLLMforAll = FrugalGPT.LLMforAll(db_path="db/HEADLINES.sqlite")

individual_rows = []
for svc in service_names:
    result = MyLLMforAll.get_completion_batch(queries=test_data, genparams=genparams, service_name=svc)
    score = FrugalGPT.compute_score(result)
    individual_rows.append({"service": svc, "accuracy": score['em'], "cost": MyLLMforAll.get_cost()})
    print(f"{svc:30s}  accuracy={score['em']:.4f}  cumulative cost=\${MyLLMforAll.get_cost():.4f}")

# get_cost() on LLMforAll accumulates across calls, so back out the per-service cost by re-instantiating
individual_rows = []
for svc in service_names:
    MyLLMforAll_single = FrugalGPT.LLMforAll(db_path="db/HEADLINES.sqlite")
    result = MyLLMforAll_single.get_completion_batch(queries=test_data, genparams=genparams, service_name=svc)
    score = FrugalGPT.compute_score(result)
    individual_rows.append({"service": svc, "accuracy": score['em'], "cost": MyLLMforAll_single.get_cost()})

import pandas as pd
individual_df = pd.DataFrame(individual_rows)
display(individual_df)


<>:17: SyntaxWarning: invalid escape sequence '\$'
<>:17: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_1884/3505417199.py:17: SyntaxWarning: invalid escape sequence '\$'
  print(f"{svc:30s}  accuracy={score['em']:.4f}  cumulative cost=\${MyLLMforAll.get_cost():.4f}")
2000it [00:03, 595.95it/s]


textsynth/gptj_6B               accuracy=0.7740  cumulative cost=\$0.0001


2000it [00:03, 596.04it/s]


textsynth/fairseq_gpt_13B       accuracy=0.7185  cumulative cost=\$0.0002


2000it [00:05, 350.67it/s]


...

openaichat/gpt-3.5-turbo        accuracy=0.8040  cumulative cost=\$0.0004


2000it [00:03, 578.15it/s]


openaichat/gpt-4                accuracy=0.8445  cumulative cost=\$0.0065


2000it [00:02, 669.65it/s]
2000it [00:03, 589.47it/s]
2000it [00:05, 347.79it/s]
2000it [00:03, 651.39it/s]
2000it [00:03, 584.16it/s]


,service,accuracy,cost
0,textsynth/gptj_6B,0.7740,0.000054
1,textsynth/fairseq_gpt_13B,0.7185,0.000158
2,ai21/j1-large,0.7555,0.000360
3,openaichat/gpt-3.5-turbo,0.8040,0.000436
4,openaichat/gpt-4,0.8445,0.006540


## 6. Train the 5-service cascade across a budget sweep

Because we now have a much wider cost range available (GPT-4 is far more expensive than the other
4 services combined), sweeping the budget traces out a longer accuracy/cost curve than the
3-provider version — it should keep extending further right along the cost axis before flattening
out near GPT-4-alone accuracy.

`cascade_depth=5` lets the cascade use up to all 5 services per query; the optimizer will still
learn to stop early (and spend less) whenever it can.

In [6]:
os.makedirs("strategy", exist_ok=True)

# Budget sweep: per-query budget, roughly log-spaced from "cheapest service only"
# up toward "GPT-4 alone" territory.
budgets = [0.0001, 0.0002, 0.001, 0.002]

CASCADE_DEPTH = 5

curve_rows = []
cascades = {}  # keep trained cascades around in case you want to inspect them later

for b in budgets:
    cas = FrugalGPT.LLMCascade()
    cas.train(
        trainingdata=train_data,
        budget=b,
        cascade_depth=CASCADE_DEPTH,
        service_names=service_names,
        genparams=genparams,
        prefix=prefix,
    )
    res = cas.get_completion_batch(queries=test_data, genparams=genparams)
    sc = FrugalGPT.compute_score(res)
    cost = cas.get_cost()
    curve_rows.append({"budget_per_query": b, "cost": cost, "accuracy": sc['em']})
    cascades[b] = cas
    print(f"budget={b:.5f}  cost=\${cost:.4f}  accuracy={sc['em']:.4f}")

curve_df = pd.DataFrame(curve_rows).sort_values("cost").reset_index(drop=True)
display(curve_df)


<>:27: SyntaxWarning: invalid escape sequence '\$'
<>:27: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_1884/3887210799.py:27: SyntaxWarning: invalid escape sequence '\$'
  print(f"budget={b:.5f}  cost=\${cost:.4f}  accuracy={sc['em']:.4f}")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CRITICAL:root:construct data for ('textsynth/gptj_6B', 'textsynth/fairseq_gpt_13B', 'ai21/j1-large', 'openaichat/gpt-3.5-turbo', 'openaichat/gpt-4')
CRITICAL:root:the obj is -1716.0 and the var is [0.15577624 0.17949359 0.17949359 0.17949359]
CRITICAL:root:construct data for ('textsynth/gptj_6B', 'textsynth/fairseq_gpt_13B', 'ai21/j1-large', 'openaichat/gpt-4', 'openaichat/gpt-3.5-turbo')
CRITICAL:root:the obj is -1692.0 and the var is [0.12821256 0.12821256 0.12821256 0.12821256]
CRITICAL:root:construct data for ('textsynth/gptj_6B', 'textsynth/fairseq_gpt_13B', 'openaichat/gpt-3.5-turbo', 'ai21/j1-large', 'openaichat/gpt-4')
CRITICAL:root:the obj is -1728.0 and the var is [0.15385308 0.15385308 0.15385308 0.15385308]
CRITICAL:root:construct data for ('textsynth/gptj_6B', 'textsynth/fairseq_gpt_13B', 'openaichat/gpt-3.5-turbo', 'openaichat/gpt-4', 'ai21/j1-large')
CRITICAL:root:the obj is -1706.0 and the var is [0.12342716 0.12894385 0.15627899 0.47568574]
CRITICAL:root:construct data

KeyboardInterrupt: 

## 7. Plot the accuracy/cost curve (5-service cascade vs. each service alone)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(curve_df["cost"], curve_df["accuracy"], "o-", color="crimson", label="FrugalGPT (5-service cascade)")

for _, row in individual_df.iterrows():
    ax.scatter([row["cost"]], [row["accuracy"]], marker="s", s=80)
    ax.annotate(row["service"].split("/")[-1], (row["cost"], row["accuracy"]),
                textcoords="offset points", xytext=(6, 4), fontsize=8)

ax.set_xlabel("Cost ($)")
ax.set_ylabel("Accuracy")
ax.set_title("FrugalGPT accuracy/cost tradeoff — HEADLINES, 5 services (2,000 train / 2,000 test)")
ax.legend()
plt.tight_layout()
plt.show()


## 8. Summary table: best cascade point vs. GPT-4 alone

In [ ]:
gpt4_row = individual_df[individual_df["service"] == "openaichat/gpt-4"].iloc[0]

# cheapest cascade budget that matches or beats GPT-4-alone accuracy
matching = curve_df[curve_df["accuracy"] >= gpt4_row["accuracy"]]
if len(matching):
    best = matching.sort_values("cost").iloc[0]
    savings = 1 - best["cost"] / gpt4_row["cost"]
    print(f"GPT-4 alone:        accuracy={gpt4_row['accuracy']:.4f}  cost=\${gpt4_row['cost']:.4f}")
    print(f"FrugalGPT (5-svc):  accuracy={best['accuracy']:.4f}  cost=\${best['cost']:.4f}")
    print(f"Cost savings to match/exceed GPT-4's accuracy: {savings*100:.1f}%")
else:
    print("No swept budget matched GPT-4-alone accuracy yet — widen the `budgets` list above (larger values) and rerun.")

summary5 = pd.concat([
    individual_df.assign(kind="individual"),
    curve_df.rename(columns={"budget_per_query": "budget"}).assign(service="FrugalGPT cascade", kind="cascade"),
], ignore_index=True, sort=False)
display(summary5)


## Notes for the writeup

- Services: **Textsynth GPT-J, Textsynth FAIRSEQ-13B, AI21 J1-Large, OpenAI GPT-3.5-turbo, OpenAI GPT-4**
  — the 5 services with (near-)full cached coverage on HEADLINES, i.e. no live API keys needed.
- `N_TRAIN=2000` / `N_EVAL=2000` — smaller than the full-dataset run, since this notebook is about
  the *shape* of the 5-service tradeoff curve rather than a Table-3-comparable single number. Raise
  both toward 10,000 the same way as the 3-provider full-scale notebook if you want that too.
- `cascade_depth=5` allows (but doesn't force) the cascade to walk through all 5 services per query;
  in practice the learned strategy should still exit early on easy queries.
- The `budgets` sweep is what produces the extended accuracy/cost curve — widen/densify that list if
  you want a smoother curve or need it to reach further toward GPT-4-alone cost.
- Same cache/no-live-key caveat as the other notebook: if `---invoke new API call---` ever prints,
  that query missed the cache.